# QuickTriage — Emergency Triage Classification Prototype

> Public benchmark used at runtime: Emergency Service Triage Application (Kaggle, 1,267 records). No source dataset or patient-level output is included in this notebook.
>
> **Educational research only. Not a diagnostic system or a clinical decision tool.**

This curated notebook keeps the main data preparation, model exploration, evaluation, SHAP, and ONNX-export workflow. Experimental duplicates and patient-specific explanation cells have been removed. Outputs and execution metadata have been cleared before publication. Kaggle credentials must be provided privately in the runtime; never commit tokens or `kaggle.json`.

## 0 — Installation des dépendances

In [ ]:
%%capture
!pip install xgboost==2.1.1 imbalanced-learn==0.12.3 shap==0.45.1 \
             onnx==1.16.1 onnxruntime==1.19.2 onnxmltools==1.12.0 \
             skl2onnx==1.17.0 kaggle plotly
print('✅ Dépendances installées')

## 1 — Téléchargement du dataset Kaggle

In [ ]:
# ── Option A : Upload manuel de kaggle.json ──────────────────────
# Si tu n'as pas encore configuré Kaggle, télécharge ton API token sur
# kaggle.com → Account → API → Create New Token  (fichier kaggle.json)
# puis exécute cette cellule pour l'uploader.

import os, json
from google.colab import files

USE_KAGGLE_API = True   # ← mettre False si tu uploades le CSV directement

if USE_KAGGLE_API:
    print('Upload ton fichier kaggle.json ...')
    uploaded = files.upload()
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    with open('/root/.config/kaggle/kaggle.json', 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
    print('✅ kaggle.json configuré')

In [ ]:
import os

if USE_KAGGLE_API:
    os.makedirs('data', exist_ok=True)
    !kaggle datasets download -d ilkeryildiz/emergency-service-triage-application -p data --unzip
    # Renommer en nom standard
    import glob
    csvs = glob.glob('data/*.csv')
    print(f'Fichiers téléchargés : {csvs}')
    RAW_PATH = csvs[0]
else:
    # ── Option B : Upload direct du CSV ──────────────────────────
    from google.colab import files
    uploaded = files.upload()
    RAW_PATH = list(uploaded.keys())[0]
    print(f'Fichier uploadé : {RAW_PATH}')

print(f'\n✅ Dataset prêt → {RAW_PATH}')

In [ ]:
import pandas as pd

df_raw = pd.read_csv(
    RAW_PATH,
    sep=';',
    encoding='Windows-1252',
    on_bad_lines='skip'
)

print(f'✅ {len(df_raw)} lignes × {len(df_raw.columns)} colonnes')
print(f'\nColonnes : {list(df_raw.columns)}')
df_raw.shape

## 2 — Exploration (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#E24B4A','#E8912B','#D4A017','#2E9E6B','#5B8FB9']

df_raw = pd.read_csv(RAW_PATH)
print(f'Dimensions : {df_raw.shape}')
print(f'Colonnes   : {list(df_raw.columns)}')
df_raw.shape

In [ ]:
# Valeurs manquantes
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing):
    print('Valeurs manquantes :')
    print(missing.to_string())
else:
    print('✅ Aucune valeur manquante')

print(f'\nTypes :\n{df_raw.dtypes}')

In [ ]:
# Distribution des niveaux KTAS
target_col = 'KTAS_expert'   # Sera détecté automatiquement ci-dessous
for col in df_raw.columns:
    if 'ktas' in col.lower() or 'expert' in col.lower():
        target_col = col
        break

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts = df_raw[target_col].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=PALETTE)
axes[0].set_title('Distribution des niveaux KTAS (avant SMOTE)', fontsize=13)
axes[0].set_xlabel('Niveau KTAS')
axes[0].set_ylabel('Nombre de patients')
for i, (k, v) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(k, v + 5, str(v), ha='center', fontsize=10)

axes[1].pie(counts.values, labels=[f'N{k}' for k in counts.index],
            colors=PALETTE, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Répartition en %', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Cible identifiée : "{target_col}"')

In [ ]:
# Corrélation signes vitaux vs niveau KTAS
vital_candidates = ['SBP','DBP','HR','RR','BT','Saturation','NRS_pain','Age']
vitals_present = [c for c in vital_candidates if c in df_raw.columns]

if vitals_present:
    corr_df = df_raw[vitals_present + [target_col]].dropna()
    corr_matrix = corr_df.corr()
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn_r',
                center=0, linewidths=0.5, square=True)
    plt.title('Corrélation signes vitaux ↔ niveau KTAS', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3 — Prétraitement

In [ ]:
# ── Configuration des features ────────────────────────────────────
# Adapter si les noms de colonnes du CSV sont différents

COLUMN_MAP = {
    'Sex':           'sex',
    'Age':           'age',
    'Arrival mode':  'arrival_mode',
    'Mental':        'mental_status',
    'NRS_pain':      'pain_score',
    'SBP':           'systolic_bp',
    'DBP':           'diastolic_bp',
    'HR':            'heart_rate',
    'RR':            'respiratory_rate',
    'BT':            'temperature',
    'Saturation':    'oxygen_saturation',
    target_col:      'ktas_level',
}

NUMERIC_FEATURES = [
    'age','temperature','heart_rate','systolic_bp',
    'diastolic_bp','oxygen_saturation','respiratory_rate','pain_score'
]
CATEGORICAL_FEATURES = ['sex','arrival_mode','mental_status']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'ktas_level'

print(f'Features numériques    : {NUMERIC_FEATURES}')
print(f'Features catégorielles : {CATEGORICAL_FEATURES}')
print(f'Total features         : {len(ALL_FEATURES)}')

In [ ]:
# ── Nettoyage ─────────────────────────────────────────────────────
df = df_raw.copy()

# Renommer
existing_map = {k: v for k, v in COLUMN_MAP.items() if k in df.columns}
df = df.rename(columns=existing_map)

# Encoder le sexe
if 'sex' in df.columns:
    df['sex'] = df['sex'].map({'Male':1,'M':1,'Female':0,'F':0,1:1,2:0}).fillna(0).astype(int)

# Supprimer lignes sans cible
df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(int)

# Ajouter features manquantes avec valeur 0
for col in ALL_FEATURES:
    if col not in df.columns:
        df[col] = 0
        print(f'  ⚠️  "{col}" absent → remplacé par 0')

# Valeurs manquantes → médiane
CLIPS = {
    'age':(0,120),'temperature':(30,43),'heart_rate':(20,250),
    'systolic_bp':(40,300),'diastolic_bp':(20,200),
    'oxygen_saturation':(50,100),'respiratory_rate':(4,60),'pain_score':(0,10)
}
for col in NUMERIC_FEATURES:
    df[col] = df[col].fillna(df[col].median())
    if col in CLIPS:
        df[col] = df[col].clip(*CLIPS[col])

for col in CATEGORICAL_FEATURES:
    df[col] = df[col].fillna(df[col].mode()[0]).astype(int)

print(f'\n✅ Dataset nettoyé : {len(df)} lignes × {len(ALL_FEATURES)} features')
df[ALL_FEATURES + [TARGET]].describe().round(2)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import pickle

os.makedirs('models', exist_ok=True)

X = df[ALL_FEATURES].values
y = df[TARGET].values - 1    # KTAS 1-5 → classes 0-4

# Split stratifié (avant SMOTE — important !)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE uniquement sur le train
print('Distribution avant SMOTE :', dict(zip(*np.unique(y_train, return_counts=True))))
sm = SMOTE(random_state=42, k_neighbors=3)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print('Distribution après SMOTE :', dict(zip(*np.unique(y_train_res, return_counts=True))))

# Normalisation (fit sur train uniquement)
scaler = StandardScaler()
n_num = len(NUMERIC_FEATURES)
X_train_res[:, :n_num] = scaler.fit_transform(X_train_res[:, :n_num])
X_test[:, :n_num]      = scaler.transform(X_test[:, :n_num])

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

import json
with open('models/feature_names.json', 'w') as f:
    json.dump(ALL_FEATURES, f)

print(f'\n✅ Train : {X_train_res.shape} | Test : {X_test.shape}')

## 4 — Entraînement XGBoost

In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)

PARAMS = {
    'n_estimators':     300,
    'max_depth':        6,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'objective':        'multi:softprob',
    'num_class':        5,
    'eval_metric':      'mlogloss',
    'random_state':     42,
    'n_jobs':           -1,
}

model = xgb.XGBClassifier(**PARAMS)

# Cross-validation stratifiée 5-fold
print('Cross-validation en cours...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring='f1_weighted')
print(f'  F1-weighted CV : {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')

# Entraînement final
model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# Sauvegarde modèle natif XGBoost
model.save_model('models/triage_xgboost.ubj')
print('\n✅ Modèle XGBoost entraîné et sauvegardé')

## 5 — Évaluation

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc          = accuracy_score(y_test, y_pred)
f1_w         = f1_score(y_test, y_pred, average='weighted')
f1_critical  = f1_score(y_test, y_pred, labels=[0,1], average='macro', zero_division=0)

KTAS_NAMES = ['N1 Critique','N2 Urgent','N3 Semi-urg.','N4 Non-urg.','N5 Mineur']

print('='*55)
print(f'  Accuracy globale       : {acc:.3f}')
print(f'  F1-weighted            : {f1_w:.3f}')
print(f'  F1 niveaux critiques   : {f1_critical:.3f}  ← priorité médicale')
print('='*55)
print()
print(classification_report(y_test, y_pred, target_names=KTAS_NAMES))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=KTAS_NAMES)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matrice de confusion', fontsize=13)
axes[0].tick_params(axis='x', rotation=30)

# Importance des features
importances = model.feature_importances_
idx = np.argsort(importances)[::-1]
axes[1].barh(
    [ALL_FEATURES[i] for i in idx],
    importances[idx],
    color='#185FA5'
)
axes[1].set_title('Importance des features (XGBoost)', fontsize=13)
axes[1].set_xlabel('Score d\'importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Courbe d'apprentissage (validation loss)
results = model.evals_result()
if results:
    epochs = len(results['validation_0']['mlogloss'])
    plt.figure(figsize=(9, 4))
    plt.plot(results['validation_0']['mlogloss'], color='#E24B4A', linewidth=1.5)
    plt.xlabel('Nombre d\'arbres')
    plt.ylabel('Log Loss (validation)')
    plt.title('Courbe d\'apprentissage', fontsize=13)
    plt.tight_layout()
    plt.show()

## 6 — Explicabilité SHAP

In [ ]:
import shap
shap.initjs()

explainer   = shap.TreeExplainer(model)
# Utiliser un sous-ensemble pour la vitesse
sample_idx  = np.random.choice(len(X_test), min(200, len(X_test)), replace=False)
shap_values = explainer.shap_values(X_test[sample_idx])

# Summary plot — vue globale toutes classes confondues
print('Impact global des features sur toutes les classes KTAS :')
shap.summary_plot(
    shap_values, X_test[sample_idx],
    feature_names=ALL_FEATURES,
    class_names=KTAS_NAMES,
    plot_type='bar',
    show=True
)

In [ ]:
# SHAP pour la classe N1 (critique) — la plus importante médicalement
print('Facteurs influençant la prédiction NIVEAU 1 (Critique) :')
shap.summary_plot(
    shap_values[0],          # classe 0 = KTAS 1
    X_test[sample_idx],
    feature_names=ALL_FEATURES,
    show=True
)

## 7 — Export ONNX (compatible FastAPI + Flutter)

In [ ]:
import onnx
import onnxmltools
from onnxmltools.convert import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

n_features = len(ALL_FEATURES)
initial_types = [('float_input', FloatTensorType([None, n_features]))]

onnx_model = convert_xgboost(
    model,
    initial_types=initial_types,
    target_opset=12,     # Compatible Android/iOS
)

onnx.save_model(onnx_model, 'models/triage_model.onnx')

# Validation
onnx.checker.check_model(onnx_model)
size_kb = os.path.getsize('models/triage_model.onnx') / 1024
print(f'✅ ONNX exporté  → models/triage_model.onnx')
print(f'   Taille        : {size_kb:.1f} KB')
print(f'   Opset         : 12 (compatible mobile)')

In [ ]:
# Vérification ONNX — tester sur un patient fictif
import onnxruntime as ort

sess = ort.InferenceSession('models/triage_model.onnx')
input_name = sess.get_inputs()[0].name

# Patient critique fictif
test_patient = np.zeros((1, n_features), dtype=np.float32)
# SpO2=88, FC=135, Tension=70 → devrait prédire N1 ou N2
feature_idx = {f: i for i, f in enumerate(ALL_FEATURES)}
test_patient[0, feature_idx['oxygen_saturation']] = 88.0
test_patient[0, feature_idx['heart_rate']]        = 135.0
test_patient[0, feature_idx['systolic_bp']]       = 70.0
test_patient[0, feature_idx['age']]               = 65.0

# Normaliser
test_patient[0, :len(NUMERIC_FEATURES)] = scaler.transform(
    test_patient[0, :len(NUMERIC_FEATURES)].reshape(1,-1)
)[0]

probs = sess.run(None, {input_name: test_patient})[0][0]
ktas  = int(np.argmax(probs)) + 1
conf  = float(np.max(probs))

print(f'Test patient critique :')
print(f'  → Niveau KTAS prédit : {ktas}')
print(f'  → Confiance          : {conf:.1%}')
print(f'  → Probabilités       : {[f"{p:.2%}" for p in probs]}')
assert ktas <= 2, '⚠️  Le modèle devrait prédire N1 ou N2 pour ce patient critique'
print('✅ Validation ONNX réussie')

## 9 — Récapitulatif & prochaines étapes

### Fichiers produits
| Fichier | Utilisation |
|---|---|
| `triage_model.onnx` | FastAPI (`api/inference.py`) + Flutter (`assets/models/`) |
| `triage_xgboost.ubj` | Réentraînement + SHAP local |
| `scaler.pkl` | Normalisation à l'inférence (copier dans `triage_ml/models/`) |
| `feature_names.json` | Validation de l'ordre des features |

### Prochaines étapes
1. Copier `models/` dans le projet `triage_ml/`
2. Lancer `uvicorn api.main:app --reload` → tester sur http://localhost:8000/docs
3. Copier `triage_model.onnx` dans `flutter_app/assets/models/`
4. Implémenter la saisie vocale et le mode hors-ligne Flutter